<div class="alert alert-info">
    <b>2.5 Full Datasets Analysis</b><br>
    Dieses Notebook analysiert die Vollständigkeit der Datensätze basierend auf verschiedenen Merkmalskombinationen.<br>
    -> <b>Ziel:</b> Ermitteln, wie viele vollständige Tupel (ohne NaN) für bestimmte Spaltenkombinationen existieren.<br>
    -> <b>Input:</b> Die 'bereinigte' Datenbank aus Schritt 2.4 (Nach IBF und Temperatur Filter).<br>
    -> <b>Output:</b> PDF-Bericht mit den Ergebnissen sortiert nach Anzahl der vollständigen Zeilen.
</div>

In [1]:
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools

In [2]:
# ------------------------- Konfiguration -------------------------
# Definition der Feature-Gruppen
GROUPS = {
    'Main Ions': ['Na', 'Ca', 'Mg', 'Cl', 'HCO3', 'SO4'],
    'Physical': ['temperature_in_c', 'pH', 'redox_potential_in_mV', 'electrical_conductivity_25c_in_uS/cm'],
    'Meta': ['rock_type', 'depth_bgl_in_m'],
    'Trace': ['Fe', 'Mn', 'Sr', 'Li', 'F', 'H2SiO3', 'NH4', 'HS', 'K', 'NO3']
}

# ----------------------- Mapping für Regex-Spalten -----------------------
ION_MAPPING = {
    'Na': 'Na_in_',
    'Ca': 'Ca_in_',
    'Mg': 'Mg_in_',
    'Cl': 'Cl_in_',
    'HCO3': 'HCO3_in_',
    'SO4': 'SO4_in_',
    'K': 'K_in_',
    'NO3': 'NO3_in_',
    'Fe': 'Fe_in_',
    'Mn': 'Mn_in_',
    'Sr': 'Sr_in_',
    'Li': 'Li_in_',
    'F': 'F_in_',
    'H2SiO3': 'H2SiO3_in_',
    'NH4': 'NH4_in_',
    'HS': 'HS_in_'
}

In [3]:
# ------------------------- Daten Laden -------------------------
base_dir = Path.cwd()
notebooks_root = base_dir.parent.parent

# ------------------------- Pfad zum Output von 2.4 (Nach Filter) -------------------------
db_path = notebooks_root / "1_Acquisition" / "1.1_Data-Acquisition-Wrapper" / "Angepasste_Datenbanken" / "Nach_IBF-und-Temp_Filter" / "Komplette_Datenbank"

# ------------------------- Finde neusten Ordner -------------------------
all_folders = [f for f in db_path.iterdir() if f.is_dir()]
latest_folder = max(all_folders, key=lambda x: x.name)
input_csv = latest_folder / "Komplette_Datenbank.csv"

print(f"Lade Daten aus: {input_csv}")
df = pd.read_csv(input_csv, low_memory=False)
print(f"Gesamtzeilen: {len(df)}")

Lade Daten aus: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_IBF-und-Temp_Filter\Komplette_Datenbank\2026-03-09_14-10-34\Komplette_Datenbank.csv
Gesamtzeilen: 94264


In [4]:
# ------------------------- Spalten zuordnen -------------------------
real_cols = {}

def find_col(partial_name):
    matches = [c for c in df.columns if partial_name in c]
    return matches[0] if matches else None

# ------------------------- Ionen mappen -------------------------
for short, partial in ION_MAPPING.items():
    found = find_col(partial)
    if found:
        real_cols[short] = found
    else:
        print(f"Warnung: Spalte für {short} nicht gefunden.")

# ------------------------- Andere Spalten direkt übernehmen  -------------------------
for group in ['Physical', 'Meta']:
    for col in GROUPS[group]:
        if col in df.columns:
            real_cols[col] = col
        elif col == 'pH':
             val = find_col('pH')
             if val: real_cols['pH'] = val
        else:
            print(f"Spalte {col} nicht gefunden.")

print("\nSpalten-Mapping abgeschlossen.")


Spalten-Mapping abgeschlossen.


In [5]:
# ------------------------- Analyse Logik (Beam Search) -------------------------
import itertools
from datetime import datetime

# ------------------------- Alle verfügbaren Spalten sammeln -------------------------
all_cols_names = GROUPS['Main Ions'] + GROUPS['Physical'] + GROUPS['Meta'] + GROUPS['Trace']
all_cols_names = sorted(list(set(all_cols_names))) # Einzigartig & Sortiert

# ------------------------- Mapping auf echte Spaltennamen -------------------------
cols_indices = [real_cols.get(c, c) for c in all_cols_names if real_cols.get(c, c) in df.columns]
col_names_clean = [c for c in all_cols_names if real_cols.get(c, c) in df.columns]
n_cols = len(cols_indices)

print(f"Anzahl Spalten im Pool: {n_cols}")

# ------------------------- Matrix erstellen (Boolean) -------------------------
df_clean = df.replace(r'^\s*$', np.nan, regex=True) 
matrix = df_clean[cols_indices].notna().values

results_by_r = {}

# ------------------------- BEAM_WIDTH: Anzahl der "besten" Kombinationen, für nächste Runde -----------------------
# Top 100, um die Top 10 der nächsten Stufe zu finden
BEAM_WIDTH = 200 
MAX_R = 15

print(f"Starte optimierte Analyse für 1 bis {MAX_R} Attribute (Beam Width: {BEAM_WIDTH})...")


# ----------------------- Liste von Tupeln (indices_tuple, count) -----------------------
current_generation = []

# ------------------------- Generation (Einzelne Spalten) -------------------------
for i in range(n_cols):
    mask = matrix[:, i]
    count = mask.sum()
    if count > 0:
        current_generation.append( ((i,), count) )
 
# ------------------------- Sortieren und Speichern -------------------------
current_generation.sort(key=lambda x: x[1], reverse=True)
top_list = current_generation[:BEAM_WIDTH]
results_by_r[1] = current_generation[:10]

for r in range(2, MAX_R + 1):
    start_time = datetime.now()
    next_generation = []
    
    candidates_checked = 0
    
    for current_indices, current_count in top_list:
        last_idx = current_indices[-1]
        

        # ------------------------- Versuche jeden möglichen Index > last_idx hinzuzufügen -------------------------
        for new_idx in range(last_idx + 1, n_cols):

            # ------------------------- Neue Kombination -------------------------
            new_indices = current_indices + (new_idx,)
            
            sub_matrix = matrix[:, new_indices]
            count = sub_matrix.all(axis=1).sum()
            
            if count > 0:
                next_generation.append( (new_indices, count) )
            
            candidates_checked += 1

    # ------------------------- Sortieren und Top-Auswahl für nächste Runde -------------------------
    next_generation.sort(key=lambda x: x[1], reverse=True)
    
    
    # ------------------------- Speichere Resultate -------------------------
    results_by_r[r] = next_generation[:10]
    top_list = next_generation[:BEAM_WIDTH]
    
    if not top_list:
        print(f"  [r={r}] Keine weiteren Kombinationen gefunden. Stoppe.")
        break
        
    print(f"  [r={r}] Fertig ({candidates_checked} geprüft). Top Count: {results_by_r[r][0][1]}")

Anzahl Spalten im Pool: 22
Starte optimierte Analyse für 1 bis 15 Attribute (Beam Width: 200)...
  [r=2] Fertig (231 geprüft). Top Count: 93957
  [r=3] Fertig (1286 geprüft). Top Count: 93379
  [r=4] Fertig (1076 geprüft). Top Count: 91606
  [r=5] Fertig (891 geprüft). Top Count: 83936
  [r=6] Fertig (769 geprüft). Top Count: 75016
  [r=7] Fertig (596 geprüft). Top Count: 60849
  [r=8] Fertig (397 geprüft). Top Count: 42985
  [r=9] Fertig (218 geprüft). Top Count: 14964
  [r=10] Fertig (177 geprüft). Top Count: 2774
  [r=11] Fertig (145 geprüft). Top Count: 725
  [r=12] Fertig (76 geprüft). Top Count: 266
  [r=13] Fertig (21 geprüft). Top Count: 155
  [r=14] Fertig (2 geprüft). Top Count: 49
  [r=15] Keine weiteren Kombinationen gefunden. Stoppe.


In [6]:
# ------------------------- Konvertierung in DataFrame-Format für Export -------------------------
final_results = {}
for r, data in results_by_r.items():
    rows = []
    for indices, count in data:
        names = [col_names_clean[i] for i in indices]
        rows.append({
            'Features': ", ".join(names),
            'Anzahl': count,
            'Prozent': (count / len(df)) * 100,
            'Fehlend': len(df) - count
        })
    final_results[r] = pd.DataFrame(rows)

print("Analyse abgeschlossen.")


# ------------------------- PDF Export -------------------------
output_pdf = "Dataset_Completeness_Global_Top10.pdf"

with PdfPages(output_pdf) as pdf:
    for r in sorted(final_results.keys()):
        subset = final_results[r]
        if subset.empty: continue
        
        fig, ax = plt.subplots(figsize=(11.69, 8.27)) # A4 Querformat
        ax.axis('tight')
        ax.axis('off')
        
        ax.set_title(f"Top 10 Kombinationen mit exakt {r} Attributen", 
                     fontsize=16, weight='bold', pad=20)
        
        # ------------------------- Formatierung -------------------------
        display_table = subset.copy()
        display_table['Prozent'] = display_table['Prozent'].apply(lambda x: f"{x:.2f}%")
        display_table['Anzahl'] = display_table['Anzahl'].apply(lambda x: f"{x:,}")
        display_table['Fehlend'] = display_table['Fehlend'].apply(lambda x: f"{x:,}")
        
        col_widths = [0.6, 0.15, 0.1, 0.15] 
        table = ax.table(cellText=display_table.values, colLabels=display_table.columns, 
                         loc='center', cellLoc='left', colWidths=col_widths)
        
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1.2, 1.8)
        
        pdf.savefig(fig, bbox_inches='tight')
        plt.close()
            
print(f"PDF gespeichert unter {Path(output_pdf).absolute()}")

Analyse abgeschlossen.
PDF gespeichert unter C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\2_Analysis\2.5_Full-Datasets-Analysis\Dataset_Completeness_Global_Top10.pdf
